### The simulations published used 300-400 CPUs for a couple of days using Julias pmap feature with workers called via SSH. The data frame that's being used is HDF5, the corresponding naming of the data one can lookup in ```save_basin_results!```. Using HDF5 allows to load data also in python efficiently.

In [ ]:
using Distributed
using Parallelism

# Code to add workers to the current network.

In [ ]:
TARGET_WORKERS = 400
#use your own code to add workers here

# Start simulation setup here, after setting up the remote worker

In [ ]:
@everywhere begin
using QuadGK
using Random
using Statistics
using LambertW
using ProgressMeter
using Distributions
end

using Plots

# Functions to save and load data

In [ ]:
# functions to save data
using HDF5

basin_data_filepath = joinpath(@__DIR__, "data")
mkpath(basin_data_filepath)
basin_data_filepath *= "/gradient_basin_data_f_run.h5"

"""
    _fitness_key(f::Float64) -> String

Stable HDF5-safe key for a Float64 fitness value (bit-exact roundtrip).
"""
@inline function _fitness_key(f::Float64)
    bits = reinterpret(UInt64, f)
    return "f_" * string(bits; base=16, pad=16)
end

"""
    _fitness_from_key(key::AbstractString) -> Float64

Inverse of `_fitness_key`.
"""
@inline function _fitness_from_key(key::AbstractString)
    startswith(key, "f_") || error("Invalid fitness key '$key'")
    bits = parse(UInt64, key[3:end]; base=16)
    return reinterpret(Float64, bits)
end

"""
    _coerce_basin_sizes(v) -> Vector{Int32}

Accept either:
- `Vector{<:Integer}`
- `Vector` of tuples where first element is basin size, e.g. `(basin_size, is_peak)`
and convert to `Vector{Int32}`.
"""
function _coerce_basin_sizes(v)
    v isa AbstractVector || error("Each dict value must be a vector of basin-size entries.")

    isempty(v) && return Int32[]

    if v isa AbstractVector{<:Integer}
        return Int32.(v)
    end

    x = first(v)
    if x isa Integer
        return Int32.(v)
    elseif x isa Tuple
        return Int32.([t[1] for t in v])
    else
        error("Unsupported basin-size value type: $(typeof(x)). Use integers or tuples with basin size first.")
    end
end

"""
    _append_to_extendable_dataset!(ds, values)

Append `values` to 1D extendable dataset `ds`.
"""
function _append_to_extendable_dataset!(ds, values::AbstractVector)
    n_new = length(values)
    n_new == 0 && return nothing

    n_old = size(ds, 1)
    n_tot = n_old + n_new
    HDF5.set_extent_dims(ds, (n_tot,))
    ds[n_old+1:n_tot] = values
    return nothing
end

"""
    save_basin_results!(L, A, fitness_to_basins; filename=basin_data_filepath)

Append basin-size samples for one `(L, A)` configuration.

Input format:
- `fitness_to_basins` is a dict-like object with
  `fitness::Float64 => basin_sizes`
- `basin_sizes` can be either:
  - a vector of integers, or
  - a vector of tuples `(basin_size, ...)` (first entry used as basin size)

Storage layout in HDF5:
- Group per config: `L{L}_A{A}`
- Subgroup: `fitness_map`
- One subgroup per fitness key (bit-exact key string), containing:
  - scalar dataset `fitness` (Float64)
  - extendable dataset `basin_sizes` (Int32)

Behavior:
- Fitness keys are processed in sorted fitness order.
- If a fitness key already exists, basin sizes are appended.
- If a fitness key is new, datasets are created.
"""
function save_basin_results!(L::Int, A::Int,
                             fitness_to_basins::AbstractDict;
                             filename::String = basin_data_filepath)
    grp_name = "L$(L)_A$(A)"

    h5open(filename, "cw") do f
        grp = haskey(f, grp_name) ? f[grp_name] : create_group(f, grp_name)
        map_grp = haskey(grp, "fitness_map") ? grp["fitness_map"] : create_group(grp, "fitness_map")

        sorted_fitness = sort(collect(keys(fitness_to_basins)))

        total_new = 0
        for fit_raw in sorted_fitness
            fit = Float64(fit_raw)
            basin_sizes = _coerce_basin_sizes(fitness_to_basins[fit_raw])
            total_new += length(basin_sizes)

            fkey = _fitness_key(fit)

            if haskey(map_grp, fkey)
                fit_grp = map_grp[fkey]
                ds_bs = fit_grp["basin_sizes"]
                n_old = size(ds_bs, 1)
                _append_to_extendable_dataset!(ds_bs, basin_sizes)
                n_new = length(basin_sizes)
                n_tot = n_old + n_new
                #println("Appended $n_new samples to '$grp_name' fitness=$fit  ($n_old -> $n_tot total)")
            else
                fit_grp = create_group(map_grp, fkey)
                fit_grp["fitness"] = fit

                n_new = length(basin_sizes)
                chunk_size = max(1, min(100_000, max(n_new, 1)))
                ds_bs = create_dataset(fit_grp, "basin_sizes", datatype(Int32),
                                       dataspace((n_new,), (-1,)); chunk=(chunk_size,))
                n_new > 0 && (ds_bs[1:n_new] = basin_sizes)

                #println("Created '$grp_name' fitness=$fit with $n_new samples")
            end
        end

        println("Finished '$grp_name': processed $(length(sorted_fitness)) fitness keys, added $total_new basin-size samples")
    end
end

"""
    load_basin_results(L, A; filename=basin_data_filepath)

Load one `(L, A)` block as a dictionary:
`Dict{Float64, Vector{Int32}}` where each key is fitness and value is all saved basin sizes.
"""
function load_basin_results(L::Int, A::Int;
                            filename::String = basin_data_filepath)
    grp_name = "L$(L)_A$(A)"

    h5open(filename, "r") do f
        haskey(f, grp_name) || error("No data for L=$L, A=$A in '$filename'")
        grp = f[grp_name]
        haskey(grp, "fitness_map") || error("Group '$grp_name' has no 'fitness_map' subgroup")

        map_grp = grp["fitness_map"]
        out = Dict{Float64, Vector{Int32}}()

        # Read and insert in sorted order for deterministic traversal.
        keys_sorted = sort(collect(keys(map_grp)); by=_fitness_from_key)
        for k in keys_sorted
            fit_grp = map_grp[k]
            fit = read(fit_grp["fitness"])
            out[fit] = read(fit_grp["basin_sizes"])
        end

        return out
    end
end

"""
    load_all_basin_results(; filename=basin_data_filepath)

Load all configs from the HDF5 file.
Returns `Dict((L, A) => Dict(fitness => basin_sizes))`.
"""
function load_all_basin_results(; filename::String = basin_data_filepath)
    results = Dict{Tuple{Int,Int}, Dict{Float64, Vector{Int32}}}()

    h5open(filename, "r") do f
        for k in keys(f)
            m = match(r"^L(\d+)_A(\d+)$", k)
            m === nothing && continue
            L, A = parse(Int, m[1]), parse(Int, m[2])

            grp = f[k]
            haskey(grp, "fitness_map") || continue
            map_grp = grp["fitness_map"]

            dict_f = Dict{Float64, Vector{Int32}}()
            keys_sorted = sort(collect(keys(map_grp)); by=_fitness_from_key)
            for fk in keys_sorted
                fit_grp = map_grp[fk]
                fit = read(fit_grp["fitness"])
                dict_f[fit] = read(fit_grp["basin_sizes"])
            end

            results[(L, A)] = dict_f
        end
    end

    return results
end

"""
    list_basin_configs(; filename=basin_data_filepath) -> Vector{Tuple{Int,Int}}

Return all `(L, A)` configs stored in the file, sorted.
"""
function list_basin_configs(; filename::String = basin_data_filepath)
    h5open(filename, "r") do f
        configs = Tuple{Int,Int}[]
        for k in keys(f)
            m = match(r"^L(\d+)_A(\d+)$", k)
            m !== nothing && push!(configs, (parse(Int, m[1]), parse(Int, m[2])))
        end
        return sort(configs)
    end
end


# Ranking of local ids (lid)

In [ ]:
@everywhere begin
function genotypes_at_distance_d_BigInt(L, A, d)
    return binomial(BigInt(L), BigInt(d)) * (BigInt(A - 1)^d)
end

function compute_d_max(L, A)
    d_max_64, d_max_128 = L, L
    
    # Int64 bounds: 0 to 2^62 - 1 (don't use the full range to be safe)
    int64_max = Int64(2)^62 - 1
    cum = 1 # due to shell 0
    for d in 0:L
        cum += genotypes_at_distance_d_BigInt(L, A, d)
        cum > int64_max && break
        d_max_64 = d
    end
    
    # Int128 bounds: 0 to 2^126 - 1 (don't use the full range to be safe)
    int128_max = Int128(2)^126 - 1
    cum = Int128(0)
    for d in d_max_64+1:L
        cum += genotypes_at_distance_d_BigInt(L, A, d)
        cum > int128_max && break
        d_max_128 = d
    end
    
    return d_max_64, d_max_128
end

struct s_d_max
    d64_max::Int64
    d128_max::Int64

    function s_d_max(L, A)
        d64_max, d128_max = compute_d_max(L, A)
        return new(d64_max, d128_max)
    end
end

#L, A = 100, 10
#cfs = s_d_max(L, A)
#d = cfs.d128_max
#binomial(Int128(100), Int128(d))*(Int128(A)-1)^d < BigInt(2)^127 - 1
end

In [ ]:
#=
# Verify d128_max is correct
L, A = 100, 20
cfs = s_d_max(L, A)
d = cfs.d128_max

# Compute safely with BigInt
count_bigint = genotypes_at_distance_d_BigInt(L, A, d)
int128_max = Int128(2)^127 - 1

println("d128_max = $d")
println("Genotype count at d=$d: $count_bigint")
println("Int128 max: $int128_max")
println("Fits in Int128? $(count_bigint <= int128_max)")

# Now check d128_max + 1
if d < L
    count_next = genotypes_at_distance_d_BigInt(L, A, d+1)
    println("Genotype count at d=$(d+1): $count_next")
    println("Would overflow Int128? $(count_next > int128_max)")
end
=#

In [ ]:
@everywhere begin

# --- --- --- --- --- --- first compute d_max for Int64 and Int128, including their offsets --- --- --- --- --- ---
function genotypes_at_distance_d_BigInt(L, A, d)
    return binomial(BigInt(L), BigInt(d)) * (BigInt(A - 1)^d)
end

function compute_d_max(L, A)
    d_max_64, d_max_128 = L, L
    
    # Int64 bounds: 0 to 2^63 - 1
    int64_max = Int64(2)^63 - 1
    n_d = 0
    for d in 0:L
        n_d = genotypes_at_distance_d_BigInt(L, A, d)
        n_d > int64_max && break
        d_max_64 = d
    end
    
    # Int128 bounds: 0 to 2^63 - 1
    int128_max = Int128(2)^127 - 1
    n_d = Int128(0)
    for d in d_max_64+1:L
        n_d = genotypes_at_distance_d_BigInt(L, A, d)
        n_d > int128_max && break
        d_max_128 = d
    end
    
    return d_max_64, d_max_128
end

struct s_d_max
    d64_max::Int64
    d128_max::Int64

    function s_d_max(L, A)
        d64_max, d128_max = compute_d_max(L, A)
        return new(d64_max, d128_max)
    end
end

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---

#Somtimes one has to use Int64 or Int128, depending on the shell size d
#Return the needed type. Note that currently no support for BigInt is implemented.
@inline function lid_type(d::Int, cfg::s_d_max)
    if d <= cfg.d64_max
        return Int64
    elseif d <= cfg.d128_max
        return Int128
    else
        error("d=$d exceeds d128_max=$(cfg.d128_max)")
    end
end

function rank_lid(
    positions::Vector{Int}, #AbstractVector{<:Integer},
    alleles::Vector{Int}, #AbstractVector{<:Integer},
    L::Int,
    A::Int,
    cfg::s_d_max
)
    d = length(positions)
    (length(alleles) == d) || error("positions and alleles must have same length")
    (0 <= d <= L) || error("d must satisfy 0 <= d <= L")
    (A >= 2) || error("A must be >= 2")
    issorted(positions) || @error("Positions need to be sorted (positions = $positions)") # check if positions are sorted

    T = lid_type(d, cfg)
    comb_rank = zero(T)
    allele_rank = zero(T)
    base = T(A - 1)
    prev = 0

    @inbounds for i in 1:d
        p = Int(positions[i])
        a = Int(alleles[i])

        for x in (prev + 1):(p - 1)
            comb_rank += binomial(T(L - x), T(d - i))
        end
        prev = p

        allele_rank = allele_rank * base + T(a - 1)
    end

    block = one(T)
    @inbounds for _ in 1:d
        block *= base
    end

    return comb_rank * block + allele_rank
end
end

# Unranking of lids

In [ ]:
@everywhere begin
"""
Inverse of the position-combination rank used in rank_lid.
Returns sorted mutation positions in 1:L.
"""
function unrank_positions(rank::T, d::Int, L::Int) where {T<:Integer}
    d < 0 && error("d must be >= 0")
    d > L && error("d must be <= L")

    total = binomial(T(L), T(d))
    (zero(T) <= rank < total) || error("combination rank out of bounds")

    r = rank
    positions = Vector{Int}(undef, d)
    prev = 0

    @inbounds for i in 1:d
        x = prev + 1
        xmax = L - (d - i)
        while x <= xmax
            c = binomial(T(L - x), T(d - i))
            if r < c
                positions[i] = x
                prev = x
                break
            end
            r -= c
            x += 1
        end
    end

    return positions
end

"""
Inverse of the allele base-(A-1) encoding used in rank_lid.
Returns allele values in 1:(A-1).
"""
function unrank_alleles(rank::T, d::Int, A::Int) where {T<:Integer}
    A >= 2 || error("A must be >= 2")
    base = T(A - 1)
    total = base^d
    (zero(T) <= rank < total) || error("allele rank out of bounds")

    r = rank
    alleles = Vector{Int}(undef, d)
    @inbounds for i in d:-1:1
        r, digit = divrem(r, base)
        alleles[i] = Int(digit + one(T))
    end
    return alleles
end

"""
Unrank shell-local lid back to (positions, alleles), matching rank_lid.
"""
function unrank_lid(lid::Integer, d::Int, L::Int, A::Int, cfg::s_d_max)
    #(0 <= d <= L) || error("d must satisfy 0 <= d <= L")
    #(A >= 2) || error("A must be >= 2")

    T = lid_type(d, cfg)
    lid_t = T(lid)
    base = T(A - 1)
    block = base^d
    total = binomial(T(L), T(d)) * block
    (zero(T) <= lid_t < total) || error("local lid out of bounds for given d")

    comb_rank = fld(lid_t, block)
    allele_rank = mod(lid_t, block)

    positions = unrank_positions(comb_rank, d, L)
    alleles = unrank_alleles(allele_rank, d, A)
    return positions, alleles
end
end

In [ ]:
#= some function to test the ranking and unranking of lids

function test_unrank_lid()
    @testset "rank/unrank roundtrip" begin
        L, A = 100, 20
        cfg = s_d_max(L, A)

        # Int64 shell
        pos1 = [2, 5]
        ale1 = [1, 2]
        lid1 = rank_lid(pos1, ale1, L, A, cfg)
        pos1_b, ale1_b = unrank_lid(lid1, length(pos1), L, A, cfg)
        println("typeof lid1: ", typeof(lid1))
        @test pos1_b == pos1
        @test ale1_b == ale1

        # Int128 shell
        pos2 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
        ale2 = [1, 2, 1, 1, 1, 1, 1, 1, 1, 1]
        lid2 = rank_lid(pos2, ale2, L, A, cfg)
        pos2_b, ale2_b = unrank_lid(lid2, length(pos2), L, A, cfg)
        println("typeof lid2: ", typeof(lid2))
        @test pos2_b == pos2
        @test ale2_b == ale2

        # Exhaustive small-shell roundtrip
        L, A, d = 5, 3, 2
        for p1 in 1:(L - 1), p2 in (p1 + 1):L, a1 in 1:(A - 1), a2 in 1:(A - 1)
            pos = [p1, p2]
            ale = [a1, a2]
            lid = rank_lid(pos, ale, L, A, cfg)
            pos_b, ale_b = unrank_lid(lid, d, L, A, cfg)
            @test pos_b == pos
            @test ale_b == ale
        end
    end

    println("All unrank_lid tests passed.")
    return nothing
end

test_unrank_lid()
=#

In [ ]:
# In compute_d_max, after finding thresholds:
# Test that rank_lid works correctly at the boundaries
#=
d_max_64, d_max_128 = compute_d_max(L, A)
positions_test = [1, 2]
alleles_test = [1, 1]
for d_test in [d_max_64, d_max_64 - 1, d_max_128, d_max_128 - 1]
    if d_test >= length(positions_test)
        lid_test = rank_lid(positions_test, alleles_test, L, A, cfg)
        if lid_test < 0  # Silent overflow wraps to negative
            error("Overflow detected in rank_lid at d=$d_test")
        end
    end
end
=#

In [ ]:
#There is the question of binomial overflows internally, so that's why we check for the baundary cases
function test_baundary_case(L, A, d, T)
    count_bigint = genotypes_at_distance_d_BigInt(L, A, d)
    T_max = T(2)^127 - 1
    binomial_T = binomial(T(L), T(d))
    allele_part_T = T(A - 1)^d
    val_T = binomial_T * allele_part_T
    return val_T == count_bigint
end

function test_both_baundary_cases(L, A)
    d_max_64, d_max_128 = compute_d_max(L, A)
    test_baundary_case(L, A, d_max_64, Int64) && test_baundary_case(L, A, d_max_128, Int128)
end


for L in 10:10:100
    for A in 2:2:20
        if test_both_baundary_cases(L, A) == false
            @error "Baundary case test failed for L=$L, A=$A"
        end
    end
end

# Function which iterates over all neighbors of a given genotype

Functions which iterate trough all neighbors of a given genotype. What's special is that they are optimized to not check all neighbors,
but only those which are needed to find the best neighbor.
This is done by keeping track of the best neighbor found so far and only checking neighbors until we find one that is better than the current best one.
This way we can skip checking all neighbors once we have found a better one.

In [ ]:
@everywhere begin
"""
Iterator over all neighboring lids of a genotype in shell d.

Usage:
    cfg = s_d_max(L, A)
    for neighbor_lid in NeighborIterator(lid, d, L, A, cfg)
        # process neighbor_lid
    end

Neighbors are yielded from shells d-1, then d, then d+1.
"""
struct NeighborIterator
    positions::Vector{Int}
    alleles::Vector{Int}
    L::Int
    A::Int
    cfg::s_d_max
    d::Int
    n_dm1::Int
    n_d::Int
    n_dp1::Int
end

# Count the number of neighbours in the shells d-1, d and d+1
function neighbor_counts_by_shell(d, L, A)
    n_dm1 = d
    n_d0  = d * (A - 2)
    n_dp1 = (L - d) * (A - 1)

    return n_dm1, n_d0, n_dp1
end

function NeighborIterator(lid::Integer, d::Int, L::Int, A::Int, cfg::s_d_max)
    (0 <= d <= L) || error("d must satisfy 0 <= d <= L")
    (A >= 2) || error("A must be >= 2")
    positions, alleles = unrank_lid(lid, d, L, A, cfg)
    n_dm1, n_d , n_dp1 = neighbor_counts_by_shell(d, L, A)
    
    return NeighborIterator(positions, alleles, L, A, cfg, d, n_dm1, n_d, n_dp1)
end

function Base.iterate(iter::NeighborIterator, idx::Int=1)
    n_total = iter.n_dm1 + iter.n_d + iter.n_dp1
    
    # Check if done
    idx > n_total && return nothing
    
    # Shell d-1: idx ∈ [1, d]
    if idx <= iter.n_dm1
        pos_in_shell = idx
        new_pos = [iter.positions[1:pos_in_shell-1]; iter.positions[pos_in_shell+1:iter.d]]
        new_ale = [iter.alleles[1:pos_in_shell-1]; iter.alleles[pos_in_shell+1:iter.d]]
        neighbor_lid = rank_lid(new_pos, new_ale, iter.L, iter.A, iter.cfg)
        return ((neighbor_lid, iter.d - 1), idx + 1)
    end
    
    # Shell d: idx ∈ [d+1, d+d*(A-2)]
    if idx <= iter.n_dm1 + iter.n_d
        pos_in_shell = idx - iter.n_dm1
        pos_idx = div(pos_in_shell - 1, iter.A - 2) + 1
        allele_offset = mod(pos_in_shell - 1, iter.A - 2)
        new_allele = allele_offset + 1
        if new_allele >= iter.alleles[pos_idx]
            new_allele += 1
        end
        new_ale = copy(iter.alleles)
        new_ale[pos_idx] = new_allele
        neighbor_lid = rank_lid(iter.positions, new_ale, iter.L, iter.A, iter.cfg)
        return ((neighbor_lid, iter.d), idx + 1)
    end
    
    # Shell d+1: idx ∈ [d+d*(A-2)+1, n_total]
    # Iterate through positions 1:L, skipping those already mutated
    pos_in_shell = idx - iter.n_dm1 - iter.n_d
    unmutated_count = 0
    
    for pos in 1:iter.L
        # Check if pos is already mutated using binary search (O(log d))
        if searchsortedfirst(iter.positions, pos) > length(iter.positions) || iter.positions[searchsortedfirst(iter.positions, pos)] != pos
            unmutated_count += 1
            if unmutated_count == div(pos_in_shell - 1, iter.A - 1) + 1
                # Found the position to add mutation at
                allele_idx = mod(pos_in_shell - 1, iter.A - 1) + 1
                insert_idx = searchsortedfirst(iter.positions, pos)
                new_pos = [iter.positions[1:insert_idx-1]; pos; iter.positions[insert_idx:iter.d]]
                new_ale = [iter.alleles[1:insert_idx-1]; allele_idx; iter.alleles[insert_idx:iter.d]]
                neighbor_lid = rank_lid(new_pos, new_ale, iter.L, iter.A, iter.cfg)
                return ((neighbor_lid, iter.d + 1), idx + 1)
            end
        end
    end
    
    return nothing
end

Base.eltype(::NeighborIterator) = Integer
end

# Functions which handle the genotype state and the coresponding dict lookups

In [ ]:
@everywhere begin
mutable struct genotype_state
    f::Float64
    best_lid::Integer #lid of neighbor with the (current) highest f value
    best_d::Int #shell of best_lid
    best_f::Float64 #fitness of the best neighbor, used to optimize the neighbor iteration
    best_k::Int #Current number of the neighbor iterator, used to optimize the neighbor iteration
    f_iter::Bool #flag which checks if all neighbors have been checked, used to optimize the neighbor iteration
end

function basin_analytical_fitness_given_fraction(n, x)
    #n-1 root of W(1) where W is the Lambert W function
    return lambertw((x-1)/n)^(1/(n - 1)) 
end
end

In [ ]:
@everywhere begin
#function that get's the state of a genotype from the dicts, given its lid and d, or creates it if it doesn't exist yet
function get_or_create_state(a_dicts, lid, d, L, A, cfg, rng, dist_f)
    state = get(a_dicts[d + 1], lid, nothing)
    if state === nothing
        #initially set itself as best neighbor, will be updated in find_best_neighbor
        f = rand(rng, dist_f)
        state = genotype_state(f, lid, d, f, 1, false)
        a_dicts[d + 1][lid] = state
    end
    return state
end

#The parent genotype is in the gradient basin.
#For one of it's neighbours (child), check if this parent has the highest fitness among the child neighbors.
#Use a shortcut: To determine if the parent has the highest fitness, one needs to check all neighbors of the child.
#But if one finds a neighbor with higher fitness than the parent, the parent will not be the path to (potentially) add it to the gradient basin.
function check_parent_is_best_neighbor(parent_state, parent_lid, parent_d,
                                       child_state, child_lid, child_d,
                                       L, A, cfg, a_dicts, rng, dist_f)
    f_found_better = false

    #if all neighbors of the child have already been checked, we can directly check if the parent is the best neighbor
    if child_state.f_iter == true
        if child_state.best_lid == parent_lid && child_state.best_d == parent_d
            return false
        else
            return true
        end
    end

    #first check if the current best neighbor of the child has a higher fitness than the parent,
    #if so we can directly return that the parent is not the best neighbor
    if child_state.best_f > parent_state.f
        return true #the parent is not the best neighbor, as the child already has a better neighbor
    end

    #we need to use a while loop, as we need to check the index of the neighbor iterator to optimize the neighbor iteration
    iter = NeighborIterator(child_lid, child_d, L, A, cfg)
    state_iter = iterate(iter, child_state.best_k)
    while state_iter !== nothing
        (neigh_lid, neigh_d), next_state = state_iter
        # ---- ---- ---- ---- ----
        #if neigh_lid == parent_lid
        #    if neigh_d == parent_d
        #        continue # skip the parent itself
        #    end
        #end
        
        neigh_state = get_or_create_state(a_dicts, neigh_lid, neigh_d, L, A, cfg, rng, dist_f)
        
        if neigh_state.f > parent_state.f #found a neighbor with higher fitness than the parent
            child_state.best_lid = neigh_lid
            child_state.best_d = neigh_d
            child_state.best_f = neigh_state.f
            child_state.best_k = next_state
            f_found_better = true
            break
        end

        # ---- ---- ---- ---- ----
        state_iter = iterate(iter, next_state)
    end

    #if all neighbors have been iterated trough, save that fact
    if state_iter == nothing
        #child_state.best_k = L*(A-1) #set best_k to the maximum number of neighbors, iteration is finished
        child_state.f_iter = true #all neighbors have been checked
    end

    #if no better neighbor was found, set the parent as best neighbor for the child
    if f_found_better == false
        child_state.best_lid = parent_lid
        child_state.best_d = parent_d
    end

    return f_found_better
end
end

In [ ]:
@everywhere begin
#initialize the dicts and the first entry for the all-zero genotype
function init_basin_dicts(L, A, f0)
    cfg = s_d_max(L, A)
    d_max_64, d_max_128 = compute_d_max(L, A)
    
    # Create shell-indexed dicts
    a_dicts = Union{Dict{Int64, genotype_state}, Dict{Int128, genotype_state}}[Dict{Int64, genotype_state}() for d in 0:d_max_64]
    d_max_128 > d_max_64 && append!(a_dicts, [Dict{Int128, genotype_state}() for d in (d_max_64+1):d_max_128])
    
    # Initialize d=0 (all-zero genotype)
    a_dicts[1][0] = genotype_state(f0, 0, 0, f0, 1, false)
    
    return a_dicts, cfg
end

function total_genotypes_created(a_dicts)
    total = 0
    for dict in a_dicts
        total += length(dict)
    end
    return total
end


"""
BFS-based gradient basin explorer with early-stopping neighbor check.

A gradient basin: each genotype flows only to its neighbor with highest fitness.
Key optimization: stop neighbor iteration once a better neighbor is found.

Args:
  - L, A: landscape parameters
  - f_func: (positions, alleles) -> Float64
  - max_genotypes: safety limit for exploration

Returns:
  - a_dicts: shell-indexed dicts storing genotype_state for each lid
  - basin_root_lid: highest-fitness genotype
"""
function calc_gradient_basin(L::Int, A::Int, f0::Float64, rng, dist_f)
    a_dicts, cfg = init_basin_dicts(L, A, f0)
    #queue = Tuple{Integer, Int, Integer, Int}[(0, 0)] #queue stores (parent_lid, parent_d, child_lid, child_d) pairs to explore
    gradient_basin = Tuple{Integer, Int}[(0, 0)] #stores (lid, d) pairs of genotypes in the gradient basin

    # Initial state for the gradient basin is the all-zero genotype
    # Checks for the members of the gradient basin (which have not been checked yet)
    # which of their children are in the gradient basin, by checking if the parent is the best neighbor of the child.

    c = 1 #counter to keep track of the neighborhood of which member of the gradient basin we are checking
    while length(gradient_basin) >= c
        lid, d = gradient_basin[c]
        #state = a_dicts[d + 1][lid] # before we added the neighbor genotype to the queue is was created, so it must be in the dicts
        
        state = get_or_create_state(a_dicts, lid, d, L, A, cfg, rng, dist_f)
        #check neighborhood of the current member of the gradient basin
        for (neigh_lid, neigh_d) in NeighborIterator(lid, d, L, A, cfg)
            neigh_state = get_or_create_state(a_dicts, neigh_lid, neigh_d, L, A, cfg, rng, dist_f)

            f_found_better = check_parent_is_best_neighbor(state, lid, d,
                                                           neigh_state, neigh_lid, neigh_d,
                                                            L, A, cfg, a_dicts, rng, dist_f)
            if f_found_better == false
                #if the parent is the best neighbor, add the child to the gradient basin
                push!(gradient_basin, (neigh_lid, neigh_d))
            end

        end

        c += 1
    end

    #for statistical purposes, checks how many genotypes were created in total during the exploration of the gradient basin
    n_total_genotypes_created = total_genotypes_created(a_dicts)

    #checks if the all genotype is a peak
    f_is_peak = check_is_peak_f0(a_dicts, 0, 0, L, A, cfg, rng, dist_f)

    #there is a potential problem with the garbage collection in julia, which causes the processes to use up too much memory.
    #Thats why we need to empty the dict, which contain a lot of information
    #for dict in a_dicts # this here is not a good code, as it blocks the memory for the worker permanently.
    #    empty!(dict) # Clears elements but reuses the memory capacity
    #end
    
    return gradient_basin, f_is_peak, n_total_genotypes_created
end

function check_is_peak_f0(a_dicts, lid, d, L, A, cfg, rng, dist_f)
    state = a_dicts[d + 1][lid]
    iter = NeighborIterator(lid, d, L, A, cfg)
    for (neigh_lid, neigh_d) in iter
        neigh_state = get_or_create_state(a_dicts, neigh_lid, neigh_d, L, A, cfg, rng, dist_f)
        if neigh_state.f > state.f
            return false #found a neighbor with higher fitness than the all-zero genotype
        end
    end
    return true #no neighbor has higher fitness than the all-zero genotype, it is a peak
end

function calc_gradient_basin_size(L::Int, A::Int, f0::Float64, rng, dist_f)
    r = calc_gradient_basin(L, A, f0, rng, dist_f)
    return length(r[1]), r[2], r[3]
end

function calc_gradient_basin_size_batch(L::Int, A::Int, a_fitness::Vector{Float64}, rng, dist_f)
    N_samp = length(a_fitness)
    a_basin_sizes = Vector{Int}(undef, N_samp)
    a_is_peak = Vector{Bool}(undef, N_samp)

    for (i, f0) in enumerate(a_fitness)
        b, p, n = calc_gradient_basin_size(L, A, f0, rng, dist_f)
        a_basin_sizes[i] = b
        a_is_peak[i] = p
    end

    #not sure if this is needed, but the notebook closed unexpectedly a few times during the testing,
    #so I added this to try to prevent it from happening again due to potential memory issues.
    GC.gc() # Perform a quick minor memory garbage collection

    return a_basin_sizes, a_fitness, a_is_peak
end
end

In [ ]:
function get_remaining_fitness_samples(L, A, iter_ω_s, N_per_bin_samples)
    # Attempt to load existing data
    existing_data = try
        load_basin_results(L, A)
    catch
        Dict{Float64, Vector{Int32}}() # Return an empty dict if the group doesn't exist
    end

    a_missing_fitness = Float64[]
    
    # Check how many samples we still need per fitness bin
    for ω in iter_ω_s
        existing_count = length(get(existing_data, ω, Int32[]))
        missing_count = max(0, N_per_bin_samples - existing_count)
        
        for _ in 1:missing_count
            push!(a_missing_fitness, ω)
        end
    end
    
    return a_missing_fitness
end

In [ ]:
configs = [(L, A) for L in 10:10:100 for A in (2, 3, 4, 10, 20)]
#configs = [(L, A) for L in 10:10:20 for A in (2, 3, 4, 10, 20)] #for testing purposed use small system sizes
sort!(configs, by = x -> x[1] * (x[2] - 1)) #sort configs by number of neighbors, which is the main driver of runtime

#needed for robust_pmap to work together with ProgressMeter
ProgressMeter.ncalls(::typeof(robust_pmap), f::Function, args...) =
    ProgressMeter.ncalls(pmap, f, args...)

# Keep only workers that still answer RPC and remove dead ones from the cluster.
function cleanup_dead_workers!(; verbose=true)
    current = workers()
    alive = Int[]

    for pid in current
        ok = try
            remotecall_fetch(() -> true, pid)
            true
        catch
            false
        end
        ok && push!(alive, pid)
    end

    dead = setdiff(current, alive)
    if !isempty(dead)
        verbose && println("Removing dead workers: $(dead)")
        try
            rmprocs(dead)
        catch err
            verbose && println("Failed to remove some dead workers: $err")
        end
    end

    return alive
end

try
    for (L, A) in configs
        n = L * (A - 1)
        b_skip = 2

        N_bins = 100
        N_per_bin_samples = 20000
        N_samp = 500 #batch size for individual workers

        rng = Random.default_rng()
        ω_s = basin_analytical_fitness_given_fraction(n, b_skip)
        dist_f0 = Uniform(ω_s, 1)
        f0 = 1.0 #rand(rng, dist_f0)

        #create fitness values for which we want to calculate the gradient basin size, which are uniformly distributed between ω_s and 1
        ϵ = 0.0000001 #need an ϵ to avoid the upper bound of 1, which would cause issues with the uniform distribution
        Δbins = (1-ϵ-ω_s) / (N_bins-1)
        iter_ω_s = ω_s:Δbins:1-ϵ

        a_fitness = get_remaining_fitness_samples(L, A, iter_ω_s, N_per_bin_samples)

        N_totalSamp = length(a_fitness)
        if N_totalSamp == 0
            #println("L=$L | A=$A: All $N_per_bin_samples samples per bin are already computed. Skipping.")
            continue
        end

        #fill_workers!(TARGET_WORKERS)

        a_perm = randperm(rng, length(a_fitness)) #do this in order to make pmap more efficient
        a_fitness = a_fitness[a_perm]

        N_batches = Int(N_totalSamp / N_samp)
        a_fitness_batches = [a_fitness[(i-1)*N_samp + 1:i*N_samp] for i in 1:N_batches]

        dist_f = Uniform(0.0, 1.0)
        λ_func = x -> calc_gradient_basin_size_batch(L, A, x, rng, dist_f)

        res = @showprogress "L=$L | A=$A | W:$(nworkers())" robust_pmap(
                    λ_func,
                    a_fitness_batches;
                    num_retries=100
                )

        # Unpack the accumulated successful results
        basin_sizes = vcat((r[1] for r in res)...)
        fitness     = vcat((r[2] for r in res)...)
        is_peak     = vcat((r[3] for r in res)...)

        dict_fitness = Dict{Float64, Vector{Tuple{Int, Bool}}}()
        for i in 1:length(fitness)
            haskey(dict_fitness, fitness[i]) == false && (dict_fitness[fitness[i]] = Tuple{Int, Bool}[])
            push!(dict_fitness[fitness[i]], (basin_sizes[i], is_peak[i]))
        end

        #println("L=$L | A=$A: Completed $N_totalSamp samples. Saving results...")

        # Save directly per (L, A). Existing fitness keys are appended in-place.
        save_basin_results!(L, A, dict_fitness)
    end
finally
    println("Cleaning up workers...")
    rmprocs(workers())
end


In [ ]:
#close all current workers to free up memory
#rmprocs(workers())